# core

> Using pyinstrument to profile FastHTML apps

Sometimes when building FastHTML apps we run into performance bottlenecks. Figuring out what is slow can be challenging, especially when building apps with async components. That's where profiling tools like pyinstrument can help. Profilers are tools that show exactly how long each component of a project takes to run. Identifying slow parts of an app is the first step in figuring out how to make things run faster.

In [1]:
#| default_exp core

In [2]:
#| export
from fasthtml.common import *
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.middleware import Middleware
try: from rich import print
except: pass
try:
    from pyinstrument import Profiler
except ImportError:
    raise ImportError('Please install pyinstrument')

In [3]:
#| export
class ProfileMiddleware(BaseHTTPMiddleware):    
    async def dispatch(self, request, call_next):
        profiling = request.query_params.get("profile", False)
        terminal = request.query_params.get("terminal", False)  
        html = request.query_params.get('write', False)
        if profiling:
            profiler = Profiler()
            profiler.start()
            response = await call_next(request)            
            profiler.stop()
            if terminal: print(profiler.output_text())
            if html: profiler.write_html('profile.html')
            return HTMLResponse(profiler.output_html())
        return await call_next(request)

## Tests

In [4]:
from starlette.testclient import TestClient
from fastcore.all import *
from httpx import ASGITransport, AsyncClient
from functools import partialmethod
from anyio import from_thread

In [5]:
class Client:
    "A simple httpx ASGI client that doesn't require `async`"
    def __init__(self, app, url="http://testserver"):
        self.cli = AsyncClient(transport=ASGITransport(app), base_url=url)

    def _sync(self, method, url, **kwargs):
        async def _request(): return await self.cli.request(method, url, **kwargs)
        with from_thread.start_blocking_portal() as portal: return portal.call(_request)

for o in ('get', 'post', 'delete', 'put', 'patch', 'options'): setattr(Client, o, partialmethod(Client._sync, o))

In [6]:
app, rt = fast_app()
app.add_middleware(ProfileMiddleware)
client = Client(app)

First, confirm that the view works normally

In [7]:
@rt
def index(): return Titled('Hello, profiler')
'Hello, profiler' in client.get('/').text

True

Now lets profile it! Or rather, check that it works.

In [8]:
'pyinstrumentHTMLRenderer' in client.get('/?profile=1').text

True

Let's print to the terminal

In [9]:
client.get('/?profile=1&terminal=1')

_     ._   __/__   _ _  _  _ _/_   Recorded: 08:44:00  Samples:  1
 /_//_/// /_\ / //_// / //_'/ //     Duration: 0.001     CPU time: 0.001
/   _/                      v5.0.1

Profile at /var/folders/3k/p8fttyg52s30bj9vdr1v7c180000gn/T/ipykernel_84995/567167746.py:12

0.001 _UnixSelectorEventLoop._run_once  asyncio/base_events.py:1909
`- 0.001 KqueueSelector.select  selectors.py:558
   `- 0.001 kqueue.control  <built-in>

<Response [200 OK]>

In [10]:
client.get('/?profile=1&write=1')

<Response [200 OK]>

In [11]:
#| hide
import nbdev; nbdev.nbdev_export()